# Genetic Coil Arena - Quick Start Tutorial

This notebook provides a hands-on introduction to evolving Snake agents using genetic algorithms.

## What You'll Learn
- How to set up and run basic evolution
- How to analyze evolution results
- How to visualize agent behavior
- How to tune hyperparameters

## 1. Setup and Imports

In [ ]:
# Install the package if needed
# !pip install -e .

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# Import genetic-coil-arena modules
from genetic_coil_arena.config import Config, load_config
from genetic_coil_arena.evo.arena import EvolutionArena
from genetic_coil_arena.snake_core.game import SnakeGame
from genetic_coil_arena.snake_core.agent import SnakeAgent

print("✅ All imports successful!")

## 2. Configuration Setup

Let's start with a basic configuration suitable for quick experimentation:

In [ ]:
# Create a configuration for quick evolution
config = Config()

# Reduce for faster experimentation
config.evolution.population_size = 50
config.evolution.generations = 20
config.evaluation.episodes_per_agent = 2
config.evaluation.parallel_processes = 4

# Game settings
config.game.board_size = 16  # Smaller board for faster games
config.game.max_steps = 200

print(f"Population size: {config.evolution.population_size}")
print(f"Generations: {config.evolution.generations}")
print(f"Board size: {config.game.board_size}x{config.game.board_size}")

## 3. Run Evolution

Now let's evolve some Snake agents!

In [ ]:
# Create evolution arena
arena = EvolutionArena(config)

print(f"Genome size: {arena.genome_size} parameters")
print(f"Observation size: {arena.observation_size}")
print("\nStarting evolution...")

# Run evolution
result = arena.evolve()

print(f"\n🎉 Evolution completed!")
print(f"Best fitness: {result.hall_of_fame[0].fitness:.4f}")
print(f"Total time: {result.total_time:.1f} seconds")

## 4. Analyze Results

Let's examine how our agents improved over time:

In [ ]:
# Extract generation data
generations = [stat.generation for stat in result.generation_stats]
best_fitness = [stat.best_fitness for stat in result.generation_stats]
mean_fitness = [stat.mean_fitness for stat in result.generation_stats]
success_rate = [stat.success_rate for stat in result.generation_stats]
diversity = [stat.diversity for stat in result.generation_stats]

# Create plots
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Evolution Progress', fontsize=16)

# Fitness evolution
axes[0, 0].plot(generations, best_fitness, 'b-', label='Best', linewidth=2)
axes[0, 0].plot(generations, mean_fitness, 'g-', label='Mean', alpha=0.7)
axes[0, 0].set_title('Fitness Over Time')
axes[0, 0].set_xlabel('Generation')
axes[0, 0].set_ylabel('Fitness')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Success rate
axes[0, 1].plot(generations, [sr * 100 for sr in success_rate], 'r-', linewidth=2)
axes[0, 1].set_title('Success Rate')
axes[0, 1].set_xlabel('Generation')
axes[0, 1].set_ylabel('Success Rate (%)')
axes[0, 1].grid(True, alpha=0.3)

# Diversity
axes[1, 0].plot(generations, diversity, 'm-', linewidth=2)
axes[1, 0].set_title('Population Diversity')
axes[1, 0].set_xlabel('Generation')
axes[1, 0].set_ylabel('Diversity')
axes[1, 0].grid(True, alpha=0.3)

# Improvement rate
fitness_improvements = np.diff(best_fitness)
axes[1, 1].plot(generations[1:], fitness_improvements, 'c-', linewidth=2)
axes[1, 1].axhline(y=0, color='k', linestyle='--', alpha=0.5)
axes[1, 1].set_title('Fitness Improvement Rate')
axes[1, 1].set_xlabel('Generation')
axes[1, 1].set_ylabel('Fitness Change')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\n📊 Evolution Summary:")
print(f"Final best fitness: {best_fitness[-1]:.4f}")
print(f"Total improvement: {best_fitness[-1] - best_fitness[0]:.4f}")
print(f"Final success rate: {success_rate[-1]:.1%}")
print(f"Average generation time: {result.total_time / len(generations):.1f} seconds")

## 5. Test the Best Agent

Let's evaluate our best evolved agent:

In [ ]:
# Get the best agent
best_genome = result.hall_of_fame[0]
best_agent = SnakeAgent.from_genome(best_genome, config.agent)

print(f"Best agent fitness: {best_genome.fitness:.4f}")

# Test the agent on multiple episodes
game = SnakeGame(
    board_size=config.game.board_size,
    observation_radius=config.game.observation_radius
)

test_episodes = 10
scores = []
lengths = []
steps_taken = []

for episode in range(test_episodes):
    state = game.reset()
    total_reward = 0
    steps = 0
    
    while state.is_alive and steps < config.game.max_steps:
        observation = game.get_observation()
        action = best_agent.predict(observation)
        state, reward, done = game.step(action)
        total_reward += reward
        steps += 1
        
        if done:
            break
    
    scores.append(state.score)
    lengths.append(state.snake_length)
    steps_taken.append(steps)
    
    print(f"Episode {episode + 1}: Score={state.score}, Length={state.snake_length}, Steps={steps}")

# Performance summary
print(f"\n🎯 Best Agent Performance ({test_episodes} episodes):")
print(f"Average score: {np.mean(scores):.1f} ± {np.std(scores):.1f}")
print(f"Best score: {max(scores)}")
print(f"Average length: {np.mean(lengths):.1f} ± {np.std(lengths):.1f}")
print(f"Success rate: {sum(1 for s in scores if s > 0) / len(scores):.1%}")
print(f"Average survival time: {np.mean(steps_taken):.1f} steps")

## 6. Compare With Random Agent

Let's see how much better our evolved agent is compared to random behavior:

In [ ]:
# Test random agent
random_scores = []
random_lengths = []

for episode in range(test_episodes):
    state = game.reset()
    steps = 0
    
    while state.is_alive and steps < config.game.max_steps:
        # Random action
        action = np.random.choice([0, 1, 2])  # straight, left, right
        state, reward, done = game.step(action)
        steps += 1
        
        if done:
            break
    
    random_scores.append(state.score)
    random_lengths.append(state.snake_length)

# Comparison plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Score comparison
ax1.bar(['Random', 'Evolved'], [np.mean(random_scores), np.mean(scores)], 
        color=['red', 'blue'], alpha=0.7)
ax1.set_title('Average Score Comparison')
ax1.set_ylabel('Average Score')
ax1.grid(True, alpha=0.3)

# Length comparison
ax2.bar(['Random', 'Evolved'], [np.mean(random_lengths), np.mean(lengths)], 
        color=['red', 'blue'], alpha=0.7)
ax2.set_title('Average Snake Length Comparison')
ax2.set_ylabel('Average Length')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate improvement
score_improvement = (np.mean(scores) / max(np.mean(random_scores), 0.1)) 
length_improvement = (np.mean(lengths) / max(np.mean(random_lengths), 1.0))

print(f"\n📈 Improvement over random:")
print(f"Score: {score_improvement:.1f}x better")
print(f"Length: {length_improvement:.1f}x better")
print(f"Random agent success rate: {sum(1 for s in random_scores if s > 0) / len(random_scores):.1%}")
print(f"Evolved agent success rate: {sum(1 for s in scores if s > 0) / len(scores):.1%}")

## 7. Hyperparameter Experiment

Let's try different mutation rates to see their effect:

In [ ]:
# Test different mutation rates
mutation_rates = [0.05, 0.1, 0.2, 0.3]
results_by_mutation = {}

for mutation_rate in mutation_rates:
    print(f"\n🧪 Testing mutation rate: {mutation_rate}")
    
    # Create config with this mutation rate
    test_config = Config()
    test_config.evolution.population_size = 30  # Even smaller for quick test
    test_config.evolution.generations = 10
    test_config.evolution.mutation_sigma = mutation_rate
    test_config.evaluation.episodes_per_agent = 1
    test_config.game.board_size = 12
    test_config.game.max_steps = 150
    
    # Run evolution
    test_arena = EvolutionArena(test_config)
    test_result = test_arena.evolve()
    
    # Store results
    final_fitness = test_result.generation_stats[-1].best_fitness
    results_by_mutation[mutation_rate] = {
        'final_fitness': final_fitness,
        'improvement': final_fitness - test_result.generation_stats[0].best_fitness,
        'time': test_result.total_time
    }
    
    print(f"Final fitness: {final_fitness:.4f}")

# Plot results
mutation_vals = list(results_by_mutation.keys())
final_fitness_vals = [results_by_mutation[m]['final_fitness'] for m in mutation_vals]
improvement_vals = [results_by_mutation[m]['improvement'] for m in mutation_vals]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(mutation_vals, final_fitness_vals, 'bo-', linewidth=2)
ax1.set_title('Final Fitness vs Mutation Rate')
ax1.set_xlabel('Mutation Rate')
ax1.set_ylabel('Final Fitness')
ax1.grid(True, alpha=0.3)

ax2.plot(mutation_vals, improvement_vals, 'go-', linewidth=2)
ax2.set_title('Total Improvement vs Mutation Rate')
ax2.set_xlabel('Mutation Rate')
ax2.set_ylabel('Fitness Improvement')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find best mutation rate
best_mutation = max(mutation_vals, key=lambda m: results_by_mutation[m]['final_fitness'])
print(f"\n🏆 Best mutation rate: {best_mutation} (fitness: {results_by_mutation[best_mutation]['final_fitness']:.4f})")

## 8. Next Steps

Congratulations! You've successfully:
- ✅ Evolved Snake agents using genetic algorithms
- ✅ Analyzed evolution progress and performance
- ✅ Compared evolved agents with random behavior
- ✅ Experimented with hyperparameter tuning

### Where to go next:

1. **Longer Evolution**: Try running with more generations and larger populations
2. **Advanced Features**: Explore novelty search, multi-objective optimization
3. **Visualization**: Use the pygame viewer to watch agents play
4. **Custom Environments**: Modify the game rules or board size
5. **Neural Architecture**: Experiment with different network architectures

### Useful commands:
```bash
# Run evolution from command line
genetic-coil evolve --generations 100 --population 200

# Analyze results
genetic-coil analyze outputs/ --plot --interactive

# Visualize best agent
genetic-coil visualize --agent outputs/hall_of_fame.pkl
```